In [1]:
%load_ext autoreload
%autoreload 2

In [25]:
import os, sys
import pandas as pd
import datetime as dt

from rockyclickup.wrapper import Session as cu_session
from rockyclickup.utils import response_to_dataframe as cu_response_to_dataframe

from rockyelevate.wrapper import Session as cu_session
from rockyelevate.utils import response_to_dataframe as elv_response_to_dataframe

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from utils.clickup import fix_clickup_date, add_client_id_to_plan_df
from utils.collector import (
    get_all_elv_organizations,
    get_all_elv_plans,
    get_all_cu_clients,
    get_all_cu_plans,
)


In [ ]:
''' GATHER CLICKUP PLANS '''
all_clickup_plans = get_all_cu_plans()
all_clickup_clients = get_all_cu_clients()

# get clickup id for each plan
cu_plan_df = add_client_id_to_plan_df(all_clickup_plans.copy())

# create rmrcode map
rmrcode_map = {
    r.get("client_id"): r.get("rmrcode")
    for i, r in all_clickup_clients.iterrows()
}

# add clients rmrcode to each plan
cu_plan_df['rmrcode'] = cu_plan_df['client_id'].map(rmrcode_map)

# fix date columns
for col in ['date_plan_start', 'date_plan_end']:
    cu_plan_df[col] = pd.to_datetime(cu_plan_df[col])
    cu_plan_df[col] = cu_plan_df[col].apply(lambda x: fix_clickup_date(x))


In [ ]:
''' GATHER ELEVATE PLANS '''
all_elevate_orgs = get_all_elv_organizations()
all_elevate_plans = get_all_elv_plans(oids=[int(x) for x in all_elevate_orgs['organization_id'].unique()])

elv_plan_df = all_elevate_plans.copy()

# convert date columns
for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    elv_plan_df[col] = pd.to_datetime(elv_plan_df[col])

# create external identifier map
external_identifier_map = {
    r.get("organization_id"): r.get("rmrcode")
    for i, r in all_elevate_orgs.iterrows()
}

# add organizations external identifier to each row
elv_plan_df['rmrcode'] = elv_plan_df['organization_id'].map(external_identifier_map)


opened 1593 organizations
opened 5912 elevate plans


In [32]:
for index, row in cu_plan_df.iterrows():
    year_from_name = row.get("name").split(" ")[-1]

    if not year_from_name.isnumeric():
        continue
    
    year_from_plan_start = row.get("date_plan_start").year
    if pd.isna(year_from_plan_start):
        continue

    if int(year_from_name) != year_from_plan_start:
        print(f"{row.get("cu_plan_id")}")
        print(f"{row.get("name")} != {dt.datetime.strftime(row.get('date_plan_start'), "%m/%d/%Y")}")



    print(year_from_name)
    break



868fraf1a
RMRMGA FSA 2025 != 01/21/1970
2025
